In [5]:
import audioflux as af
from audioflux.type import SpectralFilterBankScaleType, SpectralDataType
import numpy as np

def pca(audio_arr, sr):

    # Create BFT object and extract mel spectrogram
    bft_obj = af.BFT(
        num=128, 
        radix2_exp=12, 
        samplate=sr,  
        scale_type=SpectralFilterBankScaleType.MEL, 
        data_type=SpectralDataType.POWER  
    )

    spec_arr = bft_obj.bft(audio_arr)
    spec_arr = np.abs(spec_arr) 

    # Create XXCC object and extract mfcc
    xxcc_obj = af.XXCC(bft_obj.num)
    xxcc_obj.set_time_length(time_length=spec_arr.shape[-1])  
    mfcc_arr = xxcc_obj.xxcc(spec_arr)
   

    # Center the data (mean = 0)
    mean_vec = np.mean(mfcc_arr, axis=1, keepdims=True)
    centered_data = mfcc_arr - mean_vec

    cov = np.cov(centered_data)

    U, S, Vh = np.linalg.svd(cov)

    pca_proj = Vh[:3, :] @ centered_data

    pca_arr = pca_proj.T
    # each row is a point in 3D PCA space --> geometric trajectory

    return pca_arr